In [ ]:
#- rewrite query in better way 
# n3ml classification 
# filter extraction
#filter retrieval


In [ ]:
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("azure_collection")

In [ ]:
genai.configure(api_key=("AQ.Ab8RN6LnIFT9BEPdGHR3gUSxKw6y8xv6-_gF4oTi24_bLCKvoQ"))
model=genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
query = query = "How can I deploy a Java web application to Azure App Service?"

In [ ]:
rewrite_prompt = f"""
Rewrite the following Azure documentation search query.

Rules:
- keep it in question format.
- Keep the same meaning.
- Use Azure terminology.
- Do not answer the question.

Query:
{query}
"""

In [ ]:
response = model.generate_content(rewrite_prompt)
rewritten_query = response.text.strip()
print(query)
print(rewritten_query)

In [ ]:
classification_prompt = f"""
Classify the following Azure question into ONE category.

Categories:

Compute
Storage
Networking
Identity
Database
AI
Monitoring
Security
General
Out of scope

Question:
{rewritten_query}

Return only the category.
"""

In [ ]:
import time
print('Waiting 10 seconds for API limits...')
time.sleep(10)
classification_response = model.generate_content(classification_prompt)
classification = classification_response.text.strip()   
print(classification)

In [ ]:
parameter_prompt = f"""
Extract the following information from the Azure documentation query.


Return EXACTLY a valid JSON object with these keys:
- category
- service
- language
- topic

Query:
{rewritten_query}
"""

In [ ]:
import time
print('Waiting 10 seconds for API limits...')
time.sleep(10)
filters_response = model.generate_content(parameter_prompt)

print(filters_response.text)

In [ ]:
import json

try:
    clean_text = filters_response.text.strip()
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

    filters = json.loads(clean_text)
    print(filters)

except json.JSONDecodeError:
    print("Invalid JSON returned by Gemini:")
    print(filters_response.text)

In [ ]:
import json
# We need to parse the JSON string returned by the LLM
try:
    # Sometimes Gemini wraps JSON in markdown blocks like ```json ... ```
    raw_json = filters_response.text.strip()
    if raw_json.startswith('```json'):
        raw_json = raw_json[7:-3].strip()
    elif raw_json.startswith('```'):
        raw_json = raw_json[3:-3].strip()
        
    extracted_filters = json.loads(raw_json)
    print("Parsed Filters:", extracted_filters)
    
    # Extract a category to filter by (assuming your metadata uses 'title' or something similar)
    # Note: if your ChromaDB metadata doesn't have these exact keys, the filter might return empty.
    # This is an example of how the filter syntax works:
    category = classification
    
    print("\n--- Running Filtered Retrieval ---")
    # Assuming you want to search the rewritten query
    results = collection.query(
        query_texts=[rewritten_query],
        n_results=2
        # In a real app with proper metadata: 
        # where={"category": category}
    )
    
    print(f"\nTop Retrieved Document for '{rewritten_query}':")
    print(results['documents'][0][0][:500])
    
except Exception as e:
    print("Error parsing filters or querying DB:", e)